# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

## 2) Define Constructors Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

constructors_schema = StructType([
    StructField("constructor_id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True),
])

constructors_input_path = f"{processed_folder_path}/constructors/csv/constructors.csv"

constructors_df = spark.read \
    .option("header", True) \
    .schema(constructors_schema) \
    .csv(constructors_input_path)


constructors_dropped_df = constructors_df.drop("url")

# 3) Transform Constructors Data:

The steps included:

- Create Surrogate Key.
- Add Data Source and File Date.

In [0]:
from pyspark.sql.functions import lit

constructors_with_audit_df = constructors_dropped_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

constructors_date_df = add_ingestion_date(constructors_with_audit_df)

constructors_final_df = add_surrogate_key(
    constructors_date_df,
    key_column_name="constructor_sk",
    hash_columns=["constructor_id", "name", "nationality"],
)
 
print("Final columns going into the write:", constructors_final_df.columns)
 

# 4) Save the Processed Dataset to Delta Lake:

In [0]:

constructors_output_path = f"{processed_folder_path}/constructors/delta"
 
spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")
 
upsert_if_changed(
    input_df=constructors_final_df,
    db_name="f1_processed",
    table_name="constructors",
    output_path=constructors_output_path,
    merge_key_columns=["constructor_id"],
)

In [0]:
display(spark.read.format("delta").load(constructors_output_path))

In [0]:
build_presentation_dimension(
    processed_location=f"{processed_folder_path}/constructors/delta",
    natural_key_column="constructor_id",
    keep_columns=["constructor_id", "name", "nationality"],
    presentation_directory=f"{presentation_folder_path}/dim_constructors/delta",
    db_name="f1_presentation",
    table_name="dim_constructors",
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/dim_constructors/delta"))

# 5) Save backup Constructors in CSV format:

In [0]:
import io
import csv

constructors_backup_path = f"{presentation_folder_path}/dim_constructors/csv/dim_constructors.csv"

backup_rows = [row.asDict() for row in constructors_final_df.collect()]
backup_fieldnames = constructors_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(constructors_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {constructors_backup_path}")